Name: Anirudh Kavle
<br>
Github Username: Anirudh-Kavle
<br>
Email: kavle@usc.edu
<br>
USC ID: 6327535963

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import f_oneway

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 60)

Get the AReM Data Set

The data is in `../data/AReM`: one folder per activity, 7 folders and 88 CSV files in total. Each file is one instance, with a `time` column (a reading every 250 ms) and 6 time series: `avg_rss12, var_rss12, avg_rss13, var_rss13, avg_rss23, var_rss23`. Below, these are series 1 to 6 in that order. Every file starts with 5 comment lines beginning with `#`.

Files are sorted by activity, then by dataset number (as numbers, so `dataset10` comes after `dataset9`).

In [2]:
DATA_DIR = Path("../data/AReM")
ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
SERIES = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

def dataset_number(path):
    return int(path.stem.removeprefix("dataset"))

def load_instance(path):
    # values may be separated by commas or spaces; usecols drops the empty field left by a trailing comma
    return pd.read_csv(path, comment="#", header=None, names=["time"] + SERIES,
                       sep=r"[,\s]+", engine="python", usecols=range(7))

files = [f for act in ACTIVITIES for f in sorted((DATA_DIR / act).glob("dataset*.csv"), key=dataset_number)]
assert len(files) == 88, f"expected 88 files in {DATA_DIR}, found {len(files)}"
instances = [load_instance(f) for f in files]
meta = pd.DataFrame({"activity": [f.parent.name for f in files],
                     "dataset": [dataset_number(f) for f in files],
                     "readings": [len(df) for df in instances]},
                    index=pd.RangeIndex(1, len(files) + 1, name="Instance"))

assert not any(df.isna().any().any() for df in instances)
print(len(files), "files loaded")
print("readings per instance:", meta.readings.value_counts().to_dict())

88 files loaded
readings per instance: {480: 87, 479: 1}


Data cleaning: the check below goes through the raw text of every file, looking for formatting problems and for missing readings (each file should have one reading every 250 ms from 0 to 119,750 ms).

In [3]:
expected_times = set(range(0, 120_000, 250))
audit = []
for f, df in zip(files, instances):
    lines = [ln for ln in f.read_text().splitlines() if ln and not ln.startswith("#")]
    name = f"{f.parent.name}/{f.name}"
    if not all("," in ln for ln in lines):
        audit.append((name, "values separated by spaces, not commas"))
    n_trailing = sum(ln.rstrip().endswith(",") for ln in lines)
    if n_trailing:
        audit.append((name, f"lines ending in a comma: {n_trailing}"))
    missing = sorted(expected_times - set(df.time))
    if missing:
        audit.append((name, f"{len(df)} readings, missing {missing} ms"))
pd.DataFrame(audit, columns=["file", "problem"])

,file,problem
0,bending2/dataset4.csv,"values separated by spaces, not commas"
1,cycling/dataset9.csv,lines ending in a comma: 1
2,cycling/dataset14.csv,lines ending in a comma: 1
3,sitting/dataset8.csv,"479 readings, missing [13500] ms"


None of the input files were edited. Each problem is handled while the file is read:

- `bending2/dataset4.csv`: values are separated by spaces. The parser accepts commas or spaces, so the values are read unchanged.
- `cycling/dataset9.csv` and `cycling/dataset14.csv`: the last line (time 119,750 ms) ends with a comma, which creates an empty 8th field. Only the first 7 fields are kept.
- `sitting/dataset8.csv`: the reading at 13,500 ms is missing, so the file has 479 readings instead of 480 (484 lines counting the 5 comment lines). I didn't fill in a value. The features in 1(c) are computed from the 479 readings that are there, and none of them needs evenly spaced readings.

Some files are byte-for-byte copies of each other:

In [4]:
meta["md5"] = [hashlib.md5(f.read_bytes()).hexdigest() for f in files]
names = meta.activity + "/dataset" + meta.dataset.astype(str)
(meta.assign(file=names)[meta.md5.duplicated(keep=False)]
     .groupby("md5").file.agg(" = ".join).reset_index(drop=True).to_frame("identical files"))

,identical files
0,lying/dataset1 = lying/dataset15
1,lying/dataset9 = lying/dataset14
2,lying/dataset5 = lying/dataset12
3,lying/dataset7 = lying/dataset13
4,lying/dataset3 = lying/dataset11
5,cycling/dataset1 = cycling/dataset15


Six pairs of files are identical, so the 88 instances contain at most 82 distinct recordings. All 88 are kept, since the task defines the dataset as 88 instances. This affects the bootstrap in 1(c)iii (see the note there).

### (b) Test and Train Data

Test data: datasets 1 and 2 in `bending1` and `bending2`, and datasets 1, 2 and 3 in every other folder. All other datasets are training data.

In [5]:
test_cutoff = np.where(meta.activity.str.startswith("bending"), 2, 3)
meta["split"] = np.where(meta.dataset <= test_cutoff, "test", "train")
pd.crosstab(meta.activity, meta.split, margins=True, margins_name="Total")[["train", "test", "Total"]]

split,train,test,Total
activity,,,
bending1,5,2,7
bending2,4,2,6
cycling,12,3,15
lying,12,3,15
sitting,12,3,15
standing,12,3,15
walking,12,3,15
Total,69,19,88


### (c) Feature Extraction

#### i. Research

Time-domain features are computed directly from the values of a series, without transforming it into frequencies. Commonly used ones:

| Type | Features |
| --- | --- |
| Central value | mean, median, mode |
| Extremes | minimum, maximum, range (max - min) |
| Spread | variance, standard deviation, quartiles and other percentiles, interquartile range, mean absolute deviation, coefficient of variation |
| Shape of the distribution | skewness, kurtosis, number of values above / below the mean |
| Energy | sum of squared values (energy), root mean square, sum of values |
| Change over time | mean absolute change between consecutive values, number of peaks, number of crossings of the mean (or of zero), autocorrelation at lag k, slope of a linear trend, longest run above the mean, sample entropy |

The first five rows ignore the order of the readings; the last row depends on it. Most of these are implemented as feature calculators in the tsfresh library.

#### ii. Extraction

For each of the 6 series of each instance: minimum, maximum, mean, median, standard deviation (sample SD, divisor $n-1$), and the 1st and 3rd quartiles (25th and 75th percentiles, linear interpolation between readings). The features are used as raw values, without normalization.

Column `stat_j` is that statistic for series j. For example, `q1_6` is the first quartile of series 6 (1st quart$_6$ in the handout). The `activity`, `dataset` and `split` columns are labels, not features.

In [6]:
STATS = ["min", "max", "mean", "median", "std", "q1", "q3"]
FEATURES = [f"{stat}_{j}" for j in range(1, 7) for stat in STATS]

def extract(df):
    s = df[SERIES]
    # rows are series, so ravel gives the same series-by-series order as FEATURES
    return pd.concat([s.min(), s.max(), s.mean(), s.median(), s.std(ddof=1),
                      s.quantile(0.25), s.quantile(0.75)], axis=1).to_numpy().ravel()

features = meta[["activity", "dataset", "split"]].join(
    pd.DataFrame([extract(df) for df in instances], columns=FEATURES, index=meta.index))
print(len(features), "instances x", len(FEATURES), "features")
features.round(3)

88 instances x 42 features


,activity,dataset,split,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,min_2,max_2,mean_2,median_2,std_2,q1_2,q3_2,min_3,max_3,mean_3,median_3,std_3,q1_3,q3_3,min_4,max_4,mean_4,median_4,std_4,q1_4,q3_4,min_5,max_5,mean_5,median_5,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,bending1,1,test,37.25,45.00,40.625,40.500,1.477,39.250,42.000,0.0,1.30,0.359,0.430,0.323,0.000,0.500,4.00,29.50,19.041,19.250,4.463,16.000,23.250,0.0,7.23,0.833,0.500,0.966,0.000,1.120,27.25,38.25,34.311,35.000,2.188,33.000,36.000,0.00,1.92,0.571,0.430,0.583,0.000,1.300
2,bending1,2,test,38.00,45.67,42.813,42.500,1.436,42.000,43.670,0.0,1.22,0.372,0.470,0.289,0.000,0.500,2.00,29.50,20.097,21.000,3.894,19.000,22.250,0.0,5.76,0.876,0.500,1.047,0.000,1.145,27.67,38.50,33.025,33.000,1.995,32.000,34.500,0.00,3.11,0.571,0.430,0.601,0.000,1.300
3,bending1,3,train,35.00,47.40,43.954,44.330,1.559,43.000,45.000,0.0,1.70,0.426,0.470,0.339,0.000,0.500,6.50,29.75,22.122,23.000,3.031,19.750,24.000,0.0,4.44,0.497,0.430,0.551,0.000,0.830,29.00,38.50,35.588,36.000,2.000,35.362,36.500,0.00,1.79,0.493,0.430,0.514,0.000,0.940
4,bending1,4,train,33.00,47.75,42.180,43.500,3.671,39.150,45.000,0.0,3.00,0.696,0.500,0.631,0.000,1.120,8.50,30.00,22.184,23.000,3.810,20.500,24.372,0.0,5.15,0.990,0.830,0.954,0.430,1.300,20.00,38.67,33.494,35.000,3.849,30.458,36.330,0.00,2.18,0.614,0.500,0.524,0.000,1.000
5,bending1,5,train,33.00,45.75,41.678,41.750,2.243,41.330,42.750,0.0,2.83,0.536,0.500,0.405,0.430,0.710,3.00,28.25,19.007,19.125,4.087,16.500,22.062,0.0,6.42,0.842,0.500,0.929,0.430,1.120,23.67,37.50,29.857,30.000,2.411,28.458,31.250,0.00,1.79,0.383,0.430,0.389,0.000,0.500
6,bending1,6,train,37.00,48.00,43.455,43.250,1.386,42.500,45.000,0.0,1.58,0.378,0.470,0.316,0.000,0.500,5.75,27.00,15.793,15.000,3.848,13.000,18.270,0.0,10.03,0.849,0.500,0.996,0.430,1.120,8.00,33.50,23.035,23.500,2.489,22.250,24.000,0.00,5.26,0.680,0.500,0.623,0.430,0.870
7,bending1,7,train,36.25,48.00,43.969,44.500,1.618,43.310,44.670,0.0,1.50,0.413,0.470,0.263,0.430,0.500,1.50,26.33,15.868,16.250,3.742,14.250,18.000,0.0,5.17,0.666,0.470,0.789,0.000,0.940,11.33,30.75,22.104,21.670,3.318,20.500,23.750,0.00,2.96,0.555,0.490,0.488,0.000,0.830
8,bending2,1,test,12.75,51.00,24.563,24.250,3.738,23.188,26.500,0.0,6.87,0.591,0.430,0.837,0.000,0.710,0.00,25.33,19.121,20.250,4.300,18.000,22.000,0.0,6.76,0.737,0.470,0.927,0.000,0.870,10.67,30.00,23.493,23.750,3.694,20.500,27.000,0.00,4.97,0.700,0.500,0.694,0.430,0.870
9,bending2,2,test,0.00,42.75,27.465,28.000,3.584,25.500,30.000,0.0,7.76,0.450,0.430,0.767,0.000,0.500,7.50,35.00,20.843,20.750,3.826,18.000,22.500,0.0,5.76,0.780,0.500,0.904,0.000,0.940,0.00,33.00,17.618,18.000,5.054,15.000,20.750,0.00,6.76,1.122,0.830,1.012,0.470,1.300


#### iii. Standard Deviation

For each of the 42 features, the estimate is its sample standard deviation across the 88 instances. The 90% confidence interval comes from a percentile bootstrap:

1. Draw 88 instances with replacement, keeping each instance's 42 features together.
2. Compute the SD of every feature on that resample.
3. Repeat B = 10,000 times (seed 42).
4. The 5th and 95th percentiles of the 10,000 bootstrap SDs give the 90% interval.

In [7]:
X = features[FEATURES].to_numpy()
rng = np.random.default_rng(42)
B = 10_000
boot_sd = np.array([X[rng.integers(0, len(X), len(X))].std(axis=0, ddof=1) for _ in range(B)])

sd_table = pd.DataFrame({"SD": X.std(axis=0, ddof=1),
                         "90% CI lower": np.percentile(boot_sd, 5, axis=0),
                         "90% CI upper": np.percentile(boot_sd, 95, axis=0)}, index=FEATURES)
print("instances with a nonzero value:", (features[["min_2", "min_4", "min_6"]] > 0).sum().to_dict())
sd_table.round(3)

instances with a nonzero value: {'min_2': 0, 'min_4': 0, 'min_6': 1}


,SD,90% CI lower,90% CI upper
min_1,9.570,8.236,10.774
max_1,4.394,3.340,5.293
mean_1,5.336,4.704,5.876
median_1,5.440,4.784,6.006
std_1,1.772,1.569,1.950
q1_1,6.154,5.574,6.649
q3_1,5.139,4.339,5.834
min_2,0.000,0.000,0.000
max_2,5.063,4.631,5.401
mean_2,1.574,1.396,1.706


Notes on the intervals:

- `min_2` and `min_4` are 0 in every instance, so their SD and interval are exactly 0.
- `min_6` is nonzero in only one instance. Every resample that leaves that instance out gives SD = 0, so its interval says little.
- The bootstrap treats the 88 instances as independent. Six of them duplicate other files (see (a)), so there are at most 82 distinct recordings, and the intervals are probably slightly too narrow.

#### iv. Select Features

To pick the features I only used the 69 training instances, so the test labels play no part in the choice. The table shows two things for each feature and series:

- ANOVA F: variation between the 7 activities divided by variation within them. Higher means the feature separates the activities better.
- |corr| with mean: absolute correlation with the mean of the same series. Close to 1 means the feature mostly repeats the mean.

`min_2` and `min_4` are blank because they are constant.

In [8]:
train = features[features.split == "train"]
groups = [g for _, g in train.groupby("activity")]
scores = {}
for c in FEATURES:
    if train[c].nunique() == 1:
        continue
    stat, j = c.rsplit("_", 1)
    scores[(stat, j)] = {"ANOVA F": f_oneway(*[g[c] for g in groups]).statistic,
                         "|corr| with mean": abs(train[c].corr(train[f"mean_{j}"]))}
evidence = pd.DataFrame(scores).T.rename_axis(["feature", "series"])
evidence.unstack("series").reindex(STATS).round(2)

ANOVA F                                       |corr| with mean        \
series        1       2      3       4      5       6                1     2   
feature                                                                        
min       10.97     NaN  22.71     NaN  24.10    0.78             0.72   NaN   
max        1.51   47.73  31.69   29.33  36.95   32.96             0.55  0.89   
mean      15.85  900.16  26.36  651.76  33.43  581.45             1.00  1.00   
median    17.49  477.01  19.41  800.79  28.64  563.13             0.97  0.99   
std        9.10  360.89  15.22  128.62   7.78  133.39             0.45  0.98   
q1        21.38  265.46  21.19  244.82  25.49  346.34             0.97  0.98   
q3         8.54  807.05  28.11  561.90  32.24  664.78             0.94  1.00   

                                 
series      3     4     5     6  
feature                          
min      0.64   NaN  0.84  0.18  
max      0.84  0.83  0.78  0.83  
mean     1.00  1.00  1.00  1.00  
median   0.99  1.00  0.99  1.00  
std      0.05  0.97  0.22  0.97  
q1       0.97  0.99  0.98  0.99  
q3       0.97  1.00  0.98  1.00

I picked mean, max and standard deviation.

- mean is the only feature in the top three by ANOVA F on all six series.
- median, q1 and q3 separate the activities about as well as mean, but they correlate 0.94-1.00 with mean on every series. Next to mean they would mostly repeat it.
- max has the highest F of all seven features on series 3 and 5, and is less tied to mean (correlation 0.55-0.89). Its weak spot is series 1 (F = 1.51).
- std over min for the third pick was a close call. min has the higher F on series 1, 3 and 5, but it carries almost nothing on series 2, 4 and 6: `min_2` and `min_4` are 0 in every instance, and `min_6` is nonzero in only one (see 1(c)iii). std has no constant columns. On series 1, 3 and 5 its correlation with mean is low (0.05-0.45), so it adds something mean doesn't have; on series 2, 4 and 6 it mostly repeats mean (0.97-0.98).

## 2. ISLR 3.7.4

### (a) Linear Train

We'd expect the cubic regression's training RSS to be lower (it can't be higher). Least squares picks the coefficients that make the training RSS as small as possible. The linear model is just the cubic model with $\beta_2 = \beta_3 = 0$, so the cubic fit can always do at least as well. The two would only be equal if the fitted $\beta_2$ and $\beta_3$ came out exactly 0, which practically never happens with noisy data, since the extra terms bend to fit some of the noise $\epsilon$.

### (b) Linear Test

We'd expect the linear regression's test RSS to be lower. The true relationship is linear, so the $X^2$ and $X^3$ terms don't reduce bias. All they add is variance, because what they pick up in the training data is noise. That noise won't show up again in new data, so on average the cubic model predicts new observations worse. On any single test set the cubic could still win by chance.

### (c) Not Linear Train

We'd expect the cubic regression's training RSS to be lower. The argument in (a) doesn't depend on what the true relationship is: the cubic model contains the linear one, so its training RSS can never be higher. And since the truth isn't linear here, the extra terms can also pick up real curvature, not just noise.

### (d) Not Linear Testing

There isn't enough information to tell. It depends on how far from linear the true relationship is. If it's far from linear, the cubic model's lower bias outweighs its extra variance, and its test RSS will be lower. If it's close to linear, the extra variance can outweigh the small drop in bias, and the linear model can end up with the lower test RSS.

## References

- https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html
- https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html

*AI use: Claude Code (Anthropic) was used to edit this notebook from my own script and notes*
